In [17]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict
from typing import Annotated
from dotenv import load_dotenv
import os 
from langchain_groq import ChatGroq


load_dotenv()

True

In [18]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [19]:
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.9)

In [20]:
class State(TypedDict):
    messages:Annotated[list, add_messages]

In [21]:
schema="epm"
database="epm"
table_name="group_monitoring"
columns=[
                {
                    "name": "time",
                    "type": "timestamp",
                    "original_type": "DateTime",
                    "is_nullable": False,
                    "default_value": "",
                    "is_primary_key": True,
                    "is_numeric": False
                },
                {
                    "name": "groupName",
                    "type": "string",
                    "original_type": "LowCardinality(String)",
                    "is_nullable": False,
                    "default_value": "",
                    "is_primary_key": False,
                    "is_numeric": False
                },
                {
                    "name": "cameraId",
                    "type": "string",
                    "original_type": "String",
                    "is_nullable": False,
                    "default_value": "",
                    "is_primary_key": True,
                    "is_numeric": False
                },
                {
                    "name": "location",
                    "type": "string",
                    "original_type": "LowCardinality(String)",
                    "is_nullable": False,
                    "default_value": "",
                    "is_primary_key": True,
                    "is_numeric": False
                },
                {
                    "name": "areaName",
                    "type": "string",
                    "original_type": "LowCardinality(String)",
                    "is_nullable": False,
                    "default_value": "",
                    "is_primary_key": True,
                    "is_numeric": False
                },
                {
                    "name": "category",
                    "type": "string",
                    "original_type": "LowCardinality(String)",
                    "is_nullable": False,
                    "default_value": "",
                    "is_primary_key": False,
                    "is_numeric": False
                },
                {
                    "name": "subCategory",
                    "type": "string",
                    "original_type": "LowCardinality(String)",
                    "is_nullable": False,
                    "default_value": "",
                    "is_primary_key": False,
                    "is_numeric": False
                },
                {
                    "name": "label",
                    "type": "string",
                    "original_type": "LowCardinality(String)",
                    "is_nullable": False,
                    "default_value": "",
                    "is_primary_key": True,
                    "is_numeric": False
                },
                {
                    "name": "latitude",
                    "type": "float",
                    "original_type": "Float64",
                    "is_nullable": False,
                    "default_value": "",
                    "is_primary_key": False,
                    "is_numeric": True
                },
                {
                    "name": "longitude",
                    "type": "float",
                    "original_type": "Float64",
                    "is_nullable": False,
                    "default_value": "",
                    "is_primary_key": False,
                    "is_numeric": True
                },
                {
                    "name": "count",
                    "type": "integer",
                    "original_type": "UInt32",
                    "is_nullable": False,
                    "default_value": "",
                    "is_primary_key": False,
                    "is_numeric": True
                },
                {
                    "name": "date",
                    "type": "timestamp",
                    "original_type": "Date",
                    "is_nullable": False,
                    "default_value": "toDate(time)",
                    "is_primary_key": False,
                    "is_numeric": False
                },
                {
                    "name": "__total_count",
                    "type": "integer",
                    "original_type": "aggregate_function",
                    "is_nullable": False,
                    "default_value": None,
                    "is_primary_key": False,
                    "is_numeric": True,
                    "isGenerated": True
                }
            ]

In [22]:
from langchain_core.messages import SystemMessage, HumanMessage


prompt = """
You are an expert SQL query generator specialized in ClickHouse.

## Datasource
- **Engine**: ClickHouse (SQL dialect)
- **Database**: {database}
- **Table**: {table_name}
- **Columns**: {columns}

## Task
Convert the user's natural language request into a valid ClickHouse SQL query.

## Rules
1. Return **only** the raw SQL query — no explanations, no markdown, no code fences.
2. Always apply `LIMIT 100` unless the user explicitly requests a different limit.
3. Always `ORDER BY time DESC` unless the user specifies a different sort order.
4. Use ClickHouse-specific syntax where applicable (e.g., `toDate()`, `toDateTime()`, `dateDiff()`).
5. **ONLY use column names from `{columns}`** — never guess, infer, or hallucinate column names.
   Map user's natural language to the closest matching column name from the provided list.
6. Always qualify the table as `{database}.{table_name}` — never use the bare table name.
7. Prefer `WHERE` filters over `HAVING` unless aggregation filtering is required.
8. **STRICTLY READ-ONLY**: Never generate `INSERT`, `UPDATE`, `DELETE`, `DROP`, `ALTER`,
   `TRUNCATE`, or any other data-modifying / schema-modifying statement.
   If the user asks for one, respond with: "Only SELECT queries are supported."

## Output Format
Return a single SELECT query on `{database}.{table_name}`. Nothing else.
"""

sys_msg = SystemMessage(content=prompt.format(
    database=schema,
    table_name=table_name,
    columns=columns
))

def texttosql(state:State):
    return {"messages":llm.invoke([sys_msg]+state["messages"])}

In [23]:
workflow = StateGraph(State)

workflow.add_node("texttosql", texttosql)

workflow.add_edge(START, "texttosql")
workflow.add_edge("texttosql", END)

In [24]:
graph  = workflow.compile()

messages = [HumanMessage(content='Show hourly trend of counts for camera "CAM_005" over the last 3 days')]

messages = graph.invoke({"messages":messages})

for m in messages["messages"]:
    m.pretty_print()

================================ Human Message =================================

Show hourly trend of counts for camera "CAM_005" over the last 3 days
================================== Ai Message ==================================

SELECT toStartOfHour(time) AS hour, sum(count) AS total_count
FROM epm.group_monitoring
WHERE cameraId = 'CAM_005' AND time >= now() - INTERVAL 3 DAY
GROUP BY hour
ORDER BY hour DESC
LIMIT 100
